In [1]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import Xception
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
from tensorflow.keras.optimizers import Adam


1. PERSIAPAN DATASET


In [2]:
dataset_dir = 'dataset_daging'

# Setup Data Augmentation
train_datagen = ImageDataGenerator(
    rescale=1./255,          
    rotation_range=30,       
    width_shift_range=0.2,   
    height_shift_range=0.2,
    zoom_range=0.2,          
    horizontal_flip=True,    
    validation_split=0.2     
)

# Muat Data Training (80%)
print("Memuat data training:")
train_generator = train_datagen.flow_from_directory(
    dataset_dir,
    target_size=(299, 299),  
    batch_size=32,
    class_mode='binary',     # Sapi vs Babi = 2 kelas (binary)
    subset='training'
)

# Muat Data Validasi (20%)
print("Memuat data validasi:")
validation_generator = train_datagen.flow_from_directory(
    dataset_dir,
    target_size=(299, 299),
    batch_size=32,
    class_mode='binary',
    subset='validation'
)

Memuat data training:
Found 1212 images belonging to 2 classes.
Memuat data validasi:
Found 301 images belonging to 2 classes.


2. MEMBANGUN ARSITEKTUR XCEPTION

In [3]:
# Memuat model dasar Xception
base_model = Xception(weights='imagenet', include_top=False, input_shape=(299, 299, 3))

# Menerapkan Fine-Tuning: Membuka 20 layer terakhir Xception agar ikut belajar
for layer in base_model.layers[:-20]:
    layer.trainable = False
for layer in base_model.layers[-20:]:
    layer.trainable = True

# Menambahkan custom layer untuk klasifikasi akhir
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(256, activation='relu')(x)
x = Dropout(0.5)(x) 
predictions = Dense(1, activation='sigmoid')(x) # 1 Neuron untuk output (Sapi=0/Babi=1 atau sebaliknya)

# Menggabungkan model
model = Model(inputs=base_model.input, outputs=predictions)

# Kompilasi Model
model.compile(
    optimizer=Adam(learning_rate=0.0001), 
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# Menampilkan struktur model
print("\nRingkasan Arsitektur Model:")
model.summary()


Ringkasan Arsitektur Model:


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 299, 299,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1_conv1        │ (None, 149, 149,  │        864 │ input_layer[0][0] │
│ (Conv2D)            │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1_conv1_bn     │ (None, 149, 149,  │        128 │ block1_conv1[0][… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1_conv1_act    │ (None, 149, 149,  │          0 │ block1_conv1_bn[… │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1_conv2        │ (None, 147, 147,  │     18,432 │ block1_conv1_act… │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1_conv2_bn     │ (None, 147, 147,  │        256 │ block1_conv2[0][… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1_conv2_act    │ (None, 147, 147,  │          0 │ block1_conv2_bn[… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2_sepconv1     │ (None, 147, 147,  │      8,768 │ block1_conv2_act… │
│ (SeparableConv2D)   │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2_sepconv1_bn  │ (None, 147, 147,  │        512 │ block2_sepconv1[… │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2_sepconv2_act │ (None, 147, 147,  │          0 │ block2_sepconv1_… │
│ (Activation)        │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2_sepconv2     │ (None, 147, 147,  │     17,536 │ block2_sepconv2_… │
│ (SeparableConv2D)   │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2_sepconv2_bn  │ (None, 147, 147,  │        512 │ block2_sepconv2[… │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 74, 74,    │      8,192 │ block1_conv2_act… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2_pool         │ (None, 74, 74,    │          0 │ block2_sepconv2_… │
│ (MaxPooling2D)      │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 74, 74,    │        512 │ conv2d[0][0]      │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 74, 74,    │          0 │ block2_pool[0][0… │
│                     │ 128)              │            │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block3_sepconv1_act │ (None, 74, 74,    │          0 │ add[0][0]       

 Total params: 21,386,281 (81.58 MB)

 Trainable params: 7,851,177 (29.95 MB)

 Non-trainable params: 13,535,104 (51.63 MB)

3. PROSES TRAINING (PELATIHAN) MODEL

In [4]:
# Callback untuk menyimpan model dengan akurasi terbaik secara otomatis
checkpoint = ModelCheckpoint(
    'model_sapi_babi.h5', 
    monitor='val_accuracy', 
    save_best_only=True, 
    mode='max',
    verbose=1
)

# Callback untuk menghentikan training jika tidak ada peningkatan (mencegah overfitting)
early_stop = EarlyStopping(
    monitor='val_loss', 
    patience=5, 
    restore_best_weights=True
)

print("\nMemulai proses training...")
# Mulai melatih model (silakan sesuaikan 'epochs' jika perlu)
history = model.fit(
    train_generator,
    epochs=50, 
    validation_data=validation_generator,
    callbacks=[checkpoint, early_stop]
)

print("\nProses training selesai! Model terbaik telah disimpan dengan nama 'model_sapi_babi.h5'")


Memulai proses training...
Epoch 1/50
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 8s/step - accuracy: 0.7335 - loss: 0.5497
Epoch 1: val_accuracy improved from None to 0.72093, saving model to model_sapi_babi.h5



Epoch 1: finished saving model to model_sapi_babi.h5
38/38 ━━━━━━━━━━━━━━━━━━━━ 475s 12s/step - accuracy: 0.7335 - loss: 0.5497 - val_accuracy: 0.7209 - val_loss: 0.5612
Epoch 2/50
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 8s/step - accuracy: 0.8969 - loss: 0.2650
Epoch 2: val_accuracy improved from 0.72093 to 0.77409, saving model to model_sapi_babi.h5



Epoch 2: finished saving model to model_sapi_babi.h5
38/38 ━━━━━━━━━━━━━━━━━━━━ 445s 12s/step - accuracy: 0.8969 - loss: 0.2650 - val_accuracy: 0.7741 - val_loss: 0.4908
Epoch 3/50
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 7s/step - accuracy: 0.9505 - loss: 0.1549
Epoch 3: val_accuracy did not improve from 0.77409
38/38 ━━━━━━━━━━━━━━━━━━━━ 395s 10s/step - accuracy: 0.9505 - loss: 0.1549 - val_accuracy: 0.7076 - val_loss: 0.7212
Epoch 4/50
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 6s/step - accuracy: 0.9612 - loss: 0.1061
Epoch 4: val_accuracy did not improve from 0.77409
38/38 ━━━━━━━━━━━━━━━━━━━━ 353s 9s/step - accuracy: 0.9612 - loss: 0.1061 - val_accuracy: 0.6777 - val_loss: 0.8411
Epoch 5/50
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 6s/step - accuracy: 0.9769 - loss: 0.0723
Epoch 5: val_accuracy did not improve from 0.77409
38/38 ━━━━━━━━━━━━━━━━━━━━ 355s 9s/step - accuracy: 0.9769 - loss: 0.0723 - val_accuracy: 0.7143 - val_loss: 0.7933
Epoch 6/50
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 6s/step - accuracy: 0.9769 - loss: 0.

In [6]:
print(train_generator.class_indices)

{'Babi': 0, 'Sapi': 1}
